# EntityMatch - Quick EDA & Experiments

**Amazon ML Challenge 2026 - Business Entity Resolution**

This notebook provides quick exploratory analysis and experimentation with blocking strategies.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our modules
from utils import load_tsv, load_all_sources, parse_entity_list
from preprocessing import TextNormalizer, preprocess_dataframe

%matplotlib inline
sns.set_style('whitegrid')

print("✓ Imports successful!")

## 1. Load Sample Data

Let's start by loading a sample of the data to understand the structure.

In [ ]:
# Load just first 10k rows for quick exploration
s1_sample = pd.read_csv('../data/train/train_source1.tsv', sep='\t', nrows=10000)
s2_sample = pd.read_csv('../data/train/train_source2.tsv', sep='\t', nrows=20000)
s3_sample = pd.read_csv('../data/train/train_source3.tsv', sep='\t', nrows=20000)
gt_sample = pd.read_csv('../data/train/train_ground_truth.tsv', sep='\t', nrows=10000)

print(f"Source 1 sample: {len(s1_sample)} rows")
print(f"Source 2 sample: {len(s2_sample)} rows")
print(f"Source 3 sample: {len(s3_sample)} rows")
print(f"Ground truth sample: {len(gt_sample)} rows")

In [ ]:
# View Source 1 sample
s1_sample.head()

In [ ]:
# View ground truth
gt_sample.head()

## 2. Match Distribution Analysis

In [ ]:
# Analyze match distribution
gt_sample['match_list'] = gt_sample['matched_entity_ids'].apply(parse_entity_list)
gt_sample['num_matches'] = gt_sample['match_list'].apply(len)

# Plot distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
gt_sample['num_matches'].value_counts().sort_index().plot(kind='bar')
plt.title('Match Count Distribution (Sample)')
plt.xlabel('Number of Matches')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.pie([len(gt_sample[gt_sample['num_matches'] == 0]),
         len(gt_sample[gt_sample['num_matches'] > 0])],
        labels=['Singletons', 'With Matches'],
        autopct='%1.1f%%',
        startangle=90)
plt.title('Singleton vs Matched')

plt.tight_layout()
plt.show()

print(f"\nSingletons: {len(gt_sample[gt_sample['num_matches'] == 0])} ({len(gt_sample[gt_sample['num_matches'] == 0])/len(gt_sample)*100:.1f}%)")
print(f"Average matches: {gt_sample['num_matches'].mean():.2f}")

## 3. Text Normalization Examples

In [ ]:
# Test text normalization
test_names = [
    "ABC Corporation",
    "ABC Corp.",
    "ABC CORP",
    "XYZ Limited",
    "XYZ Ltd.",
    "Test & Company LLC"
]

print("Name Normalization Examples:")
print("-" * 50)
for name in test_names:
    normalized = TextNormalizer.normalize_business_name(name)
    print(f"{name:30} → {normalized}")

In [ ]:
# Test address normalization
test_addresses = [
    "123 Main Street, New York, NY",
    "123 Main St., New York, NY",
    "456 Oak Road, Apartment 2B",
    "456 Oak Rd., Apt 2B"
]

print("\nAddress Normalization Examples:")
print("-" * 50)
for addr in test_addresses:
    normalized = TextNormalizer.normalize_address(addr)
    print(f"{addr:40} → {normalized}")

## 4. Similarity Feature Testing

In [ ]:
from feature_engineering import FeatureExtractor
import jellyfish
from rapidfuzz import fuzz

# Test similarity functions
extractor = FeatureExtractor()

# Test pairs
test_pairs = [
    ("ABC Corporation", "ABC Corp"),
    ("ABC Corporation", "XYZ Limited"),
    ("123 Main Street", "123 Main St"),
    ("123 Main Street", "456 Oak Road")
]

print("Similarity Scores:")
print("=" * 80)
for s1, s2 in test_pairs:
    print(f"\nPair: '{s1}' vs '{s2}'")
    print(f"  Levenshtein: {extractor.levenshtein_similarity(s1, s2):.3f}")
    print(f"  Jaro-Winkler: {extractor.jaro_winkler_similarity(s1, s2):.3f}")
    print(f"  Fuzzy Ratio: {extractor.fuzzy_ratio(s1, s2):.3f}")
    print(f"  Token Sort: {extractor.fuzzy_token_sort_ratio(s1, s2):.3f}")

## 5. Blocking Strategy Experiment

Test different blocking approaches on a small sample.

In [ ]:
# Preprocess samples
s1_proc = preprocess_dataframe(s1_sample)
s2_proc = preprocess_dataframe(s2_sample)
s3_proc = preprocess_dataframe(s3_sample)

print(f"Preprocessed {len(s1_proc)} S1 records")
print(f"Preprocessed {len(s2_proc)} S2 records")
print(f"Preprocessed {len(s3_proc)} S3 records")

# Show preprocessed example
s1_proc[['entity_id', 'business_name', 'name_normalized', 'business_address', 'address_normalized']].head()

In [ ]:
# Simple phonetic blocking example
from collections import defaultdict
import jellyfish

# Build Soundex index for S2
s2_soundex_index = defaultdict(list)
for _, row in s2_proc.iterrows():
    soundex = jellyfish.soundex(row['name_normalized']) if row['name_normalized'] else ''
    s2_soundex_index[soundex].append(row['entity_id'])

print(f"Created Soundex index with {len(s2_soundex_index)} unique codes")
print(f"Average bucket size: {np.mean([len(v) for v in s2_soundex_index.values()]):.1f}")

# Test on first 100 S1 entities
total_candidates = 0
for _, row in s1_proc.head(100).iterrows():
    soundex = jellyfish.soundex(row['name_normalized']) if row['name_normalized'] else ''
    candidates = s2_soundex_index.get(soundex, [])
    total_candidates += len(candidates)

print(f"\nPhonetic blocking on 100 S1 entities:")
print(f"  Total candidates: {total_candidates}")
print(f"  Avg candidates per entity: {total_candidates/100:.1f}")
print(f"  Reduction: {(1 - total_candidates/(100*len(s2_proc)))*100:.2f}%")

## 6. F0.5 Score Understanding

Visualize how F0.5 behaves compared to F1.

In [ ]:
from utils import calculate_f05_score

# Generate precision-recall grid
precisions = np.linspace(0, 1, 50)
recalls = np.linspace(0, 1, 50)

P, R = np.meshgrid(precisions, recalls)
F05 = np.zeros_like(P)
F1 = np.zeros_like(P)

for i in range(len(precisions)):
    for j in range(len(recalls)):
        p, r = P[i, j], R[i, j]
        F05[i, j] = calculate_f05_score(p, r)
        F1[i, j] = 2*p*r/(p+r) if (p+r) > 0 else 0

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F0.5
im1 = axes[0].contourf(P, R, F05, levels=20, cmap='viridis')
axes[0].set_xlabel('Precision')
axes[0].set_ylabel('Recall')
axes[0].set_title('F0.5 Score (Precision-Heavy)')
plt.colorbar(im1, ax=axes[0])

# F1
im2 = axes[1].contourf(P, R, F1, levels=20, cmap='viridis')
axes[1].set_xlabel('Precision')
axes[1].set_ylabel('Recall')
axes[1].set_title('F1 Score (Balanced)')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("Key Observation:")
print("F0.5 contours are more horizontal → precision matters more than recall")
print("\nExample: Precision=0.8, Recall=0.6")
print(f"  F0.5 = {calculate_f05_score(0.8, 0.6):.3f}")
print(f"  F1   = {2*0.8*0.6/(0.8+0.6):.3f}")
print("\nExample: Precision=0.6, Recall=0.8")
print(f"  F0.5 = {calculate_f05_score(0.6, 0.8):.3f}")
print(f"  F1   = {2*0.6*0.8/(0.6+0.8):.3f}")

## 7. Country Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (df, name) in enumerate([(s1_sample, 'Source 1'), 
                                    (s2_sample, 'Source 2'),
                                    (s3_sample, 'Source 3')]):
    df['country'].value_counts().plot(kind='bar', ax=axes[idx])
    axes[idx].set_title(name)
    axes[idx].set_xlabel('Country')
    axes[idx].set_ylabel('Count')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Next Steps

**To run the full pipeline:**

```bash
cd ..
python src/pipeline.py --mode both --data-dir data --output-dir output
```

**Key experiments to try:**
1. Tune blocking thresholds in `src/blocking.py`
2. Add more similarity features in `src/feature_engineering.py`
3. Experiment with different models (LightGBM, CatBoost)
4. Optimize F0.5 threshold on validation set
5. Analyze error cases (false positives vs false negatives)